# TrimCI_Flow — Full End-to-End Execution

**Only two inputs:** `data/fcidump_cycle_6` and `data/dets.npz`. Nothing pre-computed.

Pipeline:
```
fcidump_cycle_6  ─┬─▶  Phase C (uncoupled)       → 118 dets  [regression check]
dets.npz          │
                  └─▶  build γ from ref det
                          │
                          ▼
                       Phase D2 (MFA h1diag)      → E_total, dets
```

The gamma (1-RDM) used to dress the fragment Hamiltonians in Phase D2 is computed
directly from the correlated reference determinant in `dets.npz` row 0 — no
pre-converged files loaded.

| | |
|---|---|
| **System** | Fe4S4, 36 orbitals, 27α + 27β electrons |
| **Target** | −327.1920 Ha, 10,095 dets (brute-force TrimCI) |

---
## 1. Setup

In [1]:
import sys, os, tempfile
from pathlib import Path
import numpy as np

REPO = Path(".").resolve()
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))

FCIDUMP  = str(REPO / "data" / "fcidump_cycle_6")
DETS_NPZ = str(REPO / "data" / "dets.npz")

REFERENCE_ENERGY = -327.1920
REFERENCE_DETS   = 10_095

assert Path(FCIDUMP).exists(),  f"FCIDUMP not found: {FCIDUMP}"
assert Path(DETS_NPZ).exists(), f"dets.npz not found: {DETS_NPZ}"
print("Inputs:")
print(f"  FCIDUMP  : {FCIDUMP}")
print(f"  dets.npz : {DETS_NPZ}")

Inputs:
  FCIDUMP  : /home/unfunnypanda/Proj_Flow/TrimCI_Flow/data/fcidump_cycle_6
  dets.npz : /home/unfunnypanda/Proj_Flow/TrimCI_Flow/data/dets.npz


---
## 2. Build γ from scratch

The mean-field environment for Phase D2 is a diagonal 1-RDM (occupation vector).
We derive it directly from the correlated reference determinant stored in `dets.npz` row 0:

$$\gamma_{pp} = (\alpha_{\text{ref}} >> p) \& 1 \;+\; (\beta_{\text{ref}} >> p) \& 1 \quad \in \{0, 1, 2\}$$

No pre-computed files. This is the starting-point 1-RDM for the MFA dressing.

In [2]:
import trimci

# Read system parameters from FCIDUMP
h1, eri, n_elec, n_orb, E_nuc, n_alpha, n_beta, psym = trimci.read_fcidump(FCIDUMP)
print(f"FCIDUMP: n_orb={n_orb}, n_elec={n_elec} ({n_alpha}α + {n_beta}β), E_nuc={E_nuc}")

# Load correlated reference determinant (row 0 of dets.npz)
dets_data  = np.load(DETS_NPZ)
ref_alpha  = int(dets_data["dets"][0, 0])
ref_beta   = int(dets_data["dets"][0, 1])
n_ref_dets = len(dets_data["dets"])
print(f"dets.npz: {n_ref_dets} reference dets loaded, using row 0 as correlated ref det")

# Build diagonal gamma from ref det occupation numbers
gamma_ref = np.array(
    [((ref_alpha >> p) & 1) + ((ref_beta >> p) & 1) for p in range(n_orb)],
    dtype=np.float64
)

print(f"\nγ from ref det:")
print(f"  sum = {gamma_ref.sum():.1f}  (should equal n_elec = {n_elec})")
print(f"  unique values: {sorted(set(gamma_ref.astype(int)))}  (0=empty, 1=singly occ, 2=doubly occ)")
print(f"  doubly occ orbs : {int((gamma_ref == 2).sum())}")
print(f"  singly occ orbs : {int((gamma_ref == 1).sum())}")
print(f"  empty orbs      : {int((gamma_ref == 0).sum())}")

# Save to a temp file — run_mfa_d2 expects a file path
_gamma_tmp = tempfile.NamedTemporaryFile(suffix=".npy", delete=False)
np.save(_gamma_tmp.name, gamma_ref)
GAMMA_FROM_SCRATCH = _gamma_tmp.name
print(f"\nγ saved to temp file: {GAMMA_FROM_SCRATCH}")

FCIDUMP: n_orb=36, n_elec=54 (27α + 27β), E_nuc=0.0
dets.npz: 10095 reference dets loaded, using row 0 as correlated ref det

γ from ref det:
  sum = 54.0  (should equal n_elec = 54)
  unique values: [np.int64(0), np.int64(1), np.int64(2)]  (0=empty, 1=singly occ, 2=doubly occ)
  doubly occ orbs : 19
  singly occ orbs : 16
  empty orbs      : 1

γ saved to temp file: /tmp/tmpsjdm_7au.npy


---
## 3. Phase C — Uncoupled Baseline (regression check)

Overlapping sliding-window fragments (W=15, S=10 → 3 fragments). Bare integrals, no coupling.
Regression-locked: must return **118 dets, [51, 51, 16]**.

In [3]:
from TrimCI_Flow.uncoupled import run_fragmented_trimci

print("Running Phase C (uncoupled, 3 overlapping fragments)...\n")
result_c = run_fragmented_trimci(FCIDUMP)

Running Phase C (uncoupled, 3 overlapping fragments)...



[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 32207175
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 105 entries in 0.002260s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.06, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 191, final threshold: 0.06
[Trim] Start. Pool=191
[Trim] Core-core H precomputed: 1 dets, 1 nnz, 3.9063e-05s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=11
[Trim] Final diagonalization...
[Trim] Final E=-183.1743056186
[C++] Iteration 1 energy: -183.174306
[C++] Core set: 1, Raw dets: 11
[C++] Core set: 1 -> 10 (first cycle)
[C++] Iteration 1 time: 0.002507s (Total: 0.007662s)
[C++] Iteration 2/200000
[PoolBuild] Starting pool build: target_size=200, threshold=0.0600000000, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0540000000, restarting from initial pool

[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 859, final threshold: 0.0209207064
[Trim] Start. Pool=859
[Trim] Core-core H precomputed: 42 dets, 593 nnz, 0.0006552790s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=52
[Trim] Final diagonalization...
[Trim] Final E=-187.6120801147
[C++] Iteration 34 energy: -187.612080
[C++] Core set: 42, Raw dets: 52
[C++] ΔE = 0.004014
[C++] Core set: 42 -> 43
[C++] Iteration 34 time: 0.017758s (Total: 0.218874s)
[C++] Iteration 35/200000
[PoolBuild] Starting pool build: target_size=860, threshold=0.0209207064, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 860, final threshold: 0.0209207064
[Trim] Start. Pool=860
[Trim] Core-core H precomputed: 43 dets, 622 nnz, 0.0003334690s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=53
[Trim] Final diagonalization...
[Trim] Final E=-187.6128966360
[C++] Iteration 35 energy: -187.612897
[C++] Core set: 43, Raw dets: 53
[C++] ΔE

[Trim] Start. Pool=476
[Trim] Core-core H precomputed: 22 dets, 172 nnz, 0.0003408320s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=32
[Trim] Final diagonalization...
[Trim] Final E=-219.8929138223
[C++] Iteration 14 energy: -219.892914
[C++] Core set: 22, Raw dets: 32
[C++] ΔE = -0.077858
[C++] Core set: 22 -> 23
[C++] Iteration 14 time: 0.006417s (Total: 0.047919s)
[C++] Iteration 15/200000
[PoolBuild] Starting pool build: target_size=460, threshold=0.0486000000, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0437400000, restarting from initial pool=23
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 474, final threshold: 0.0437400000
[Trim] Start. Pool=474
[Trim] Core-core H precomputed: 23 dets, 192 nnz, 0.0000538910s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=33
[Trim] Final diagonalization...
[Trim] Final E=-219.9190193147
[C++] Iteration 15 energy: -219.919019
[C++] Core set: 23, Raw dets: 33
[C++] ΔE = -0.026105
[C++] Core set

[Trim] Start. Pool=1092
[Trim] Core-core H precomputed: 40 dets, 427 nnz, 0.0108775580s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=50
[Trim] Final diagonalization...
[Trim] Final E=-220.3204658935
[C++] Iteration 32 energy: -220.320466
[C++] Core set: 40, Raw dets: 50
[C++] ΔE = -0.005829
[C++] Core set: 40 -> 41
[C++] Iteration 32 time: 0.091175s (Total: 0.314412s)
[C++] Iteration 33/200000
[PoolBuild] Starting pool build: target_size=820, threshold=0.0286978140, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1086, final threshold: 0.0286978140
[Trim] Start. Pool=1086
[Trim] Core-core H precomputed: 41 dets, 438 nnz, 0.0091488600s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=51
[Trim] Final diagonalization...
[Trim] Final E=-220.3186013176
[C++] Iteration 33 energy: -220.318601
[C++] Core set: 41, Raw dets: 51
[C++] ΔE = 0.001865
[C++] Core set: 41 -> 42
[C++] Iteration 33 time: 0.057834s (Total: 0.372251s)
[C++] Iterat

  Fragment orbs 0..35: n_dets=16, energy=-247.6590 Ha


In [4]:
print("Phase C — Results")
print("=" * 52)
for i, (n, orbs) in enumerate(zip(result_c.fragment_n_dets, result_c.fragment_orbs)):
    print(f"  Fragment {i}  orbs [{orbs[0]:2d}..{orbs[-1]:2d}]  :  {n:4d} dets")
print("-" * 52)
print(f"  Total                          :  {result_c.total_dets:4d} dets")
print(f"  Brute-force reference          :  {REFERENCE_DETS:,} dets")
print(f"  Cost fraction                  :  {result_c.total_dets / REFERENCE_DETS:.1%}")
print()
assert result_c.total_dets == 118 and result_c.fragment_n_dets == [51, 51, 16], \
    f"REGRESSION FAILED: got {result_c.fragment_n_dets}"
print("  Regression check               :  PASS ✓")

Phase C — Results
  Fragment 0  orbs [ 2..33]  :    51 dets
  Fragment 1  orbs [ 4..33]  :    51 dets
  Fragment 2  orbs [ 0..35]  :    16 dets
----------------------------------------------------
  Total                          :   118 dets
  Brute-force reference          :  10,095 dets
  Cost fraction                  :  1.2%

  Regression check               :  PASS ✓


---
## 4. Phase D2 — MFA Non-Overlapping (best model)

Non-overlapping h1-diagonal partition (3 × 12 orbitals). Each fragment's h1 is Fock-dressed
by the diagonal γ built from the reference determinant above. TrimCI solves each fragment
independently on the dressed Hamiltonian. Energy via the correlation-correction formula:

$$E_\text{total} = E_\text{MF,global} + \sum_I \bigl(E_\text{TrimCI,I} - E_\text{MF,emb,I}\bigr)$$

In [5]:
from TrimCI_Flow.mfa import run_mfa_d2

TRIMCI_CONFIG = {
    "threshold":           0.06,
    "max_final_dets":      "auto",
    "max_rounds":          2,
    "num_runs":            1,
    "pool_build_strategy": "heat_bath",
    "verbose":             False,
}

print("Running Phase D2 (MFA h1diag, threshold=0.06)...")
print("  γ source: correlated reference det (row 0 of dets.npz) — no pre-computed files\n")
result_d2 = run_mfa_d2(
    fcidump_path=FCIDUMP,
    gamma_path=GAMMA_FROM_SCRATCH,
    ref_dets_path=DETS_NPZ,
    trimci_config=TRIMCI_CONFIG,
    partition="h1diag",
)

Running Phase D2 (MFA h1diag, threshold=0.06)...
  γ source: correlated reference det (row 0 of dets.npz) — no pre-computed files



[MFA-D2] n_orb=36, n_elec=54, E_nuc=0.0
[MFA-D2] gamma_mixed: shape=(36, 36), Tr=54.0000
[MFA-D2 WARNING] gamma_source loaded as diagonal_vector_promoted_to_matrix
[MFA-D2] 3 non-overlapping fragments of 12 orbitals each (partition=h1diag)
[MFA-D2] E_mf_global = -322.715001 Ha (E_nuc=0.0)
[MFA-D2] Row-partition sum=-322.715001, E_mf_global_elec=-322.715001, match=True
[C++ Workflow] Starting with 1 initial determinants
[C++ Workflow] Total possible configurations: 853776
[C++ Workflow] Sparsity detected: h1=1.000000, eri=1.000000
[C++ Workflow] Double exc table: 66 entries in 0.000323s
[C++] Iteration 1/200000
[PoolBuild] Starting pool build: target_size=100, threshold=0.0600000000, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0540000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0486000000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0437400000, restarting from initial pool=1
[PoolBuild] threshold relaxed to 0.0393660000, re

[Trim] Start. Pool=617
[Trim] Core-core H precomputed: 15 dets, 66 nnz, 0.0000544020s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=25
[Trim] Final diagonalization...
[Trim] Final E=-26.1735475114
[C++] Iteration 7 energy: -26.173548
[C++] Core set: 15, Raw dets: 25
[C++] ΔE = -0.056387
[C++] Core set: 15 -> 16
[C++] Iteration 7 time: 0.063973s (Total: 0.225272s)
[C++] Iteration 8/200000
[PoolBuild] Starting pool build: target_size=320, threshold=0.0137260755, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 459, final threshold: 0.0137260755
[Trim] Start. Pool=459
[Trim] Core-core H precomputed: 16 dets, 73 nnz, 0.0000527660s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=26
[Trim] Final diagonalization...
[Trim] Final E=-26.1964423199
[C++] Iteration 8 energy: -26.196442
[C++] Core set: 16, Raw dets: 26
[C++] ΔE = -0.022895
[C++] Core set: 16 -> 17
[C++] Iteration 8 time: 0.004815s (Total: 0.230093s)
[C++] Iteration 9/200000


[Trim] Start. Pool=899
[Trim] Core-core H precomputed: 44 dets, 426 nnz, 0.0000845690s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=54
[Trim] Final diagonalization...
[Trim] Final E=-26.2651727294
[C++] Iteration 36 energy: -26.265173
[C++] Core set: 44, Raw dets: 54
[C++] ΔE = -0.000770
[C++] Core set: 44 -> 45
[C++] Iteration 36 time: 0.047150s (Total: 0.446307s)
[C++] Iteration 37/200000
[PoolBuild] Starting pool build: target_size=900, threshold=0.0090056781, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0081051103, restarting from initial pool=45
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 941, final threshold: 0.0081051103
[Trim] Start. Pool=941
[Trim] Core-core H precomputed: 45 dets, 438 nnz, 0.0069651060s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=55
[Trim] Final diagonalization...
[Trim] Final E=-26.2660166323
[C++] Iteration 37 energy: -26.266017
[C++] Core set: 45, Raw dets: 55
[C++] ΔE = -0.000844
[C++] Core set: 45

[PoolBuild] threshold relaxed to 0.0065651393, restarting from initial pool=61
[PoolBuild] threshold relaxed to 0.0059086254, restarting from initial pool=61
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1220, final threshold: 0.0059086254
[Trim] Start. Pool=1220
[Trim] Core-core H precomputed: 61 dets, 702 nnz, 0.0034466600s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=71
[Trim] Final diagonalization...
[Trim] Final E=-26.2696325948
[C++] Iteration 48 energy: -26.269633
[C++] Core set: 61, Raw dets: 71
[C++] ΔE = -0.000336
[C++] Core set: 61 -> 63
[C++] Iteration 48 time: 0.053440s (Total: 0.700076s)
[C++] Iteration 49/200000
[PoolBuild] Starting pool build: target_size=1260, threshold=0.0059086254, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1828, final threshold: 0.0059086254
[Trim] Start. Pool=1828
[Trim] Core-core H precomputed: 63 dets, 745 nnz, 0.0001139170s
[Trim] Round 1 (m=10, k=1)
[Trim] Ro

...

[Trim] Start. Pool=721
[Trim] Core-core H precomputed: 24 dets, 124 nnz, 0.0098859470s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=34
[Trim] Final diagonalization...
[Trim] Final E=-42.4026880064
[C++] Iteration 16 energy: -42.402688
[C++] Core set: 24, Raw dets: 34
[C++] ΔE = -0.000842
[C++] Core set: 24 -> 25
[C++] Iteration 16 time: 0.097522s (Total: 0.415826s)
[C++] Iteration 17/200000
[PoolBuild] Starting pool build: target_size=500, threshold=0.0047859866, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 729, final threshold: 0.0047859866
[Trim] Start. Pool=729
[Trim] Core-core H precomputed: 25 dets, 138 nnz, 0.0006809460s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=35
[Trim] Final diagonalization...
[Trim] Final E=-42.4023400191
[C++] Iteration 17 energy: -42.402340
[C++] Core set: 25, Raw dets: 35
[C++] ΔE = 0.000348
[C++] Core set: 25 -> 26
[C++] Iteration 17 time: 0.020328s (Total: 0.436186s)
[C++] Iteration 18/

[Trim] Start. Pool=806
[Trim] Core-core H precomputed: 40 dets, 310 nnz, 0.0024783640s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=50
[Trim] Final diagonalization...
[Trim] Final E=-42.4081296168
[C++] Iteration 32 energy: -42.408130
[C++] Core set: 40, Raw dets: 50
[C++] ΔE = -0.000199
[C++] Core set: 40 -> 41
[C++] Iteration 32 time: 0.036408s (Total: 0.644775s)
[C++] Iteration 33/200000
[PoolBuild] Starting pool build: target_size=820, threshold=0.0043073879, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0038766491, restarting from initial pool=41
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 824, final threshold: 0.0038766491
[Trim] Start. Pool=824
[Trim] Core-core H precomputed: 41 dets, 323 nnz, 0.0000992940s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=51
[Trim] Final diagonalization...
[Trim] Final E=-42.4087296732
[C++] Iteration 33 energy: -42.408730
[C++] Core set: 41, Raw dets: 51
[C++] ΔE = -0.000600
[C++] Core set: 41

[Trim] Start. Pool=1115
[Trim] Core-core H precomputed: 42 dets, 333 nnz, 0.0008943620s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=52
[Trim] Final diagonalization...
[Trim] Final E=-42.4085689840
[C++] Iteration 34 energy: -42.408569
[C++] Core set: 42, Raw dets: 52
[C++] ΔE = 0.000161
[C++] Core set: 42 -> 43
[C++] Iteration 34 time: 0.035897s (Total: 0.854504s)
[C++] Iteration 35/200000
[PoolBuild] Starting pool build: target_size=860, threshold=0.0038766491, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1108, final threshold: 0.0038766491
[Trim] Start. Pool=1108
[Trim] Core-core H precomputed: 43 dets, 357 nnz, 0.0015722410s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=53
[Trim] Final diagonalization...
[Trim] Final E=-42.4099283057
[C++] Iteration 35 energy: -42.409928
[C++] Core set: 43, Raw dets: 53
[C++] ΔE = -0.001359
[C++] Core set: 43 -> 44
[C++] Iteration 35 time: 0.039210s (Total: 0.893720s)
[C++] Iteration 

[Trim] Start. Pool=1310
[Trim] Core-core H precomputed: 53 dets, 435 nnz, 0.0091802530s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=63
[Trim] Final diagonalization...
[Trim] Final E=-42.4147212076
[C++] Iteration 44 energy: -42.414721
[C++] Core set: 53, Raw dets: 63
[C++] ΔE = -0.002080
[C++] Core set: 53 -> 55
[C++] Iteration 44 time: 0.040008s (Total: 1.082963s)
[C++] Iteration 45/200000
[PoolBuild] Starting pool build: target_size=1100, threshold=0.0034889842, mode=heat_bath, factor=1
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1194, final threshold: 0.0034889842
[Trim] Start. Pool=1194
[Trim] Core-core H precomputed: 55 dets, 484 nnz, 0.0001220980s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=65
[Trim] Final diagonalization...
[Trim] Final E=-42.4149545119
[C++] Iteration 45 energy: -42.414955
[C++] Core set: 55, Raw dets: 65
[C++] ΔE = -0.000233
[C++] Core set: 55 -> 57
[C++] Iteration 45 time: 0.006129s (Total: 1.089128s)
[C++] Iteratio

[Trim] Start. Pool=1489
[Trim] Core-core H precomputed: 69 dets, 706 nnz, 0.0109636600s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=79
[Trim] Final diagonalization...
[Trim] Final E=-42.4161433134
[C++] Iteration 52 energy: -42.416143
[C++] Core set: 69, Raw dets: 79
[C++] ΔE = -0.000160
[C++] Core set: 69 -> 71
[C++] Iteration 52 time: 0.157557s (Total: 1.315494s)
[C++] Iteration 53/200000
[PoolBuild] Starting pool build: target_size=1420, threshold=0.0031400858, mode=heat_bath, factor=1
[PoolBuild] threshold relaxed to 0.0028260772, restarting from initial pool=71
[PoolBuild] target_size reached, stopping.
[PoolBuild] Final pool size: 1515, final threshold: 0.0028260772
[Trim] Start. Pool=1515
[Trim] Core-core H precomputed: 71 dets, 740 nnz, 0.0001526740s
[Trim] Round 1 (m=10, k=1)
[Trim] Round 1 end. Pool=81
[Trim] Final diagonalization...
[Trim] Final E=-42.4162756698
[C++] Iteration 53 energy: -42.416276
[C++] Core set: 71, Raw dets: 81
[C++] ΔE = -0.000132
[C++] Core set

In [6]:
e     = result_d2["E_total"]
emf   = result_d2["E_mf_global"]
ndets = result_d2["fragment_n_dets"]
total = sum(ndets)
error = e - REFERENCE_ENERGY

print("Phase D2 — Results")
print("=" * 56)
print(f"  γ source     : ref det occupation (from scratch)")
print(f"  E_mf_global  : {emf:.6f} Ha")
print()
for i, n in enumerate(ndets):
    print(f"  Fragment {i}   : {n:4d} dets")
print(f"  Total dets   : {total}")
print()
print(f"  E_total      : {e:.6f} Ha")
print(f"  Reference    : {REFERENCE_ENERGY:.4f}   Ha")
print(f"  Error        : {error:+.4f} Ha")
print(f"  Cost         : {total / REFERENCE_DETS:.1%} of brute-force dets")

# Cleanup temp gamma file
os.unlink(GAMMA_FROM_SCRATCH)

Phase D2 — Results
  γ source     : ref det occupation (from scratch)
  E_mf_global  : -322.715001 Ha

  Fragment 0   :   73 dets
  Fragment 1   :   73 dets
  Fragment 2   :    1 dets
  Total dets   : 147

  E_total      : -327.477016 Ha
  Reference    : -327.1920   Ha
  Error        : -0.2850 Ha
  Cost         : 1.5% of brute-force dets


---
## 5. Summary

In [7]:
rows = [
    ("Brute-force TrimCI",          REFERENCE_DETS,          REFERENCE_ENERGY),
    ("Phase C  uncoupled",          result_c.total_dets,     None),
    ("Phase D2  MFA h1diag t=0.06", sum(result_d2["fragment_n_dets"]), result_d2["E_total"]),
]

print(f"{'Model':<32}  {'dets':>6}  {'% ref':>7}  {'E_total (Ha)':>14}  {'error':>10}")
print("-" * 78)
for label, dets, e in rows:
    pct   = f"{dets / REFERENCE_DETS:>7.1%}"
    e_s   = f"{e:>14.6f}" if e is not None else f"{'(overlapping)':>14}"
    err_s = f"{e - REFERENCE_ENERGY:>+10.4f} Ha" if e is not None else f"{'—':>10}"
    print(f"{label:<32}  {dets:>6}  {pct}  {e_s}  {err_s}")

print()
gap = result_d2["E_total"] - REFERENCE_ENERGY
print(f"  Remaining gap : {gap:+.4f} Ha")
print(f"  Next step     : Phase E — PT2 cross-fragment coupling (F0 ↔ F1)")

Model                               dets    % ref    E_total (Ha)       error
------------------------------------------------------------------------------
Brute-force TrimCI                 10095   100.0%     -327.192000     +0.0000 Ha
Phase C  uncoupled                   118     1.2%   (overlapping)           —
Phase D2  MFA h1diag t=0.06          147     1.5%     -327.477016     -0.2850 Ha

  Remaining gap : -0.2850 Ha
  Next step     : Phase E — PT2 cross-fragment coupling (F0 ↔ F1)
